<a href="https://colab.research.google.com/github/avyue/datasci112_finalproject/blob/main/hic_2025_shelter_beds_by_spa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import io, requests
import numpy as np
import pandas as pd
import plotly.express as px
pd.set_option("display.max_columns", 120)

In [ ]:
HIC_URL = "https://www.lahsa.org/item.ashx?id=9369-housing-inventory-count-hic-.xlsx&dl=true"

resp = requests.get(HIC_URL, headers={"User-Agent": "Mozilla/5.0"})
resp.raise_for_status()
content = resp.content
print("downloaded bytes:", len(content), "| looks like a real .xlsx:", content[:2] == b"PK")
# If 'looks like a real .xlsx' is False, the link changed: open https://www.lahsa.org/hic,
# right-click "2025 Housing Inventory Count", copy link, and paste it into HIC_URL above.

xls = pd.ExcelFile(io.BytesIO(content))
print("sheets:", xls.sheet_names)

downloaded bytes: 1330693 | looks like a real .xlsx: True
sheets: ['Cover Page', 'Codebook', 'LA CoC Summary (2025)', '2025 HIC - All Projects', '2025 HIC - Shelter Projects', '2025 HIC - PH Projects']


In [ ]:
SHEET = "2025 HIC - All Projects"
print("Using sheet:", SHEET)

# bonus: read the built-in codebook so you know what every column means
codebook = pd.read_excel(xls, sheet_name="Codebook", header=None)
print("\n--- Codebook ---")
print(codebook.to_string())

Using sheet: 2025 HIC - All Projects

--- Codebook ---
                                                0                                       1                                                                                                                                                                                                                                                                                                       2
0   2025 Housing Inventory Count (HIC) - Codebook                                     NaN                                                                                                                                                                                                                                                                                                     NaN
1                                             NaN                         HIC Column Name                                                                                    

In [ ]:
peek = pd.read_excel(xls, sheet_name=SHEET, header=None, nrows=15)

def find_header_row(frame, tokens=("project", "bed", "unit", "organization", "type", "spa")):
    for i, row in frame.iterrows():
        cells = row.astype(str).str.lower()
        if sum(cells.str.contains(t).any() for t in tokens) >= 2:
            return i
    return 0

hdr = find_header_row(peek)
print("header row index:", hdr)

df = pd.read_excel(xls, sheet_name=SHEET, header=hdr)
df.columns = df.columns.astype(str).str.strip()
print("shape:", df.shape)
df.head()

header row index: 0
shape: (1189, 79)


,Year,Proj. Type,Organization Name,Project Name,City,State,SPA,CD,SD,Geo Code,HMIS Participating,Inventory Type,Bed Type,Target Pop.,Beds HH w/ Children,Units HH w/ Children,Beds HH w/o Children,Beds HH w/ only Children,Veteran Beds HH w/ Children,Youth Beds HH w/ Children,CH Beds HH w/ Children,Veteran Beds HH w/o Children,Youth Beds HH w/o Children,CH Beds HH w/o Children,CH Beds HH w only Children,Victim Service Provider,Additional Federal Funding?,Additional Federal Funding: HUD-VASH,Additional Federal Funding: SSVF,Additional Federal Funding: GPD-BH,Additional Federal Funding: GPD-LD,Additional Federal Funding: GPD-HH,Additional Federal Funding: GPD-CT,Additional Federal Funding: GPD-SITH,Additional Federal Funding: GPD-TP,Additional Federal Funding: HCHV-CRS,Additional Federal Funding: HCHV-SH,Additional Federal Funding: BCP,Additional Federal Funding: MGH,Additional Federal Funding: TLP,Additional Federal Funding: RhyDp,Additional Federal Funding: HOPWA-HMV,Additional Federal Funding: HOPWA-PH,Additional Federal Funding: HOPWA-STSF,Additional Federal Funding: HOPWA-TH,Additional Federal Funding: HOPWA-CV,Additional Federal Funding: PIH,Additional Federal Funding: PIH-EHV,Additional Federal Funding: HOME,Additional Federal Funding: HOME-ARP,Additional Federal Funding: Other,Housing Type,McKinney- Vento,McKinney- Vento: EsgEs,McKinney- Vento: EsgRrh,McKinney- Vento: Esg-CV,McKinney- Vento: Esg-RUSH,McKinney- Vento: CocSh,McKinney- Vento: CocTh,McKinney- Vento: CocPsh,McKinney- Vento: CocRrh,McKinney- Vento: CocSro,McKinney- Vento: CocThRrh,McKinney- Vento: SpC,McKinney- Vento: S8,McKinney- Vento: SHP,McKinney- Vento: Yhdp,McKinney- Vento: YhdpRenewals,McKinney- Vento: Unshelt,McKinney- Vento: Rural,Year-Round Beds,Total Seasonal Beds,Availability Start Date,Availability End Date,Overflow Beds,PIT Count,Total Beds,Utilization Rate,Unnamed: 78
0,2025.0,ES,211,CES for Families - SPA 3 (formerly named Infor...,San Gabriel,CA,SPA 3 ...,0,1,069037,Yes,Current,Voucher beds,NaN,33.0,12.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Tenant-based - scattered site,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,33.0,0.0,NaT,NaT,0.0,33.0,33.0,1.0,NaN
1,2025.0,ES,1736 Family Crisis Center,"A Bridge Home CD 10 ""Entry/Exit",Los Angeles,CA,SPA 4 ...,10,2,062118,Yes,Current,Facility-based beds,NaN,0.0,0.0,15.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No,Yes,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,Site-based - single site,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,15.0,0.0,NaT,NaT,0.0,15.0,15.0,1.0,NaN
2,2025.0,ES,1736 Family Crisis Center,"CES Enhanced Bridge Housing for Women SPA 8 ""E...",Long Beach,CA,SPA 8 ...,0,4,062118,Yes,Current,Facility-based beds,NaN,0.0,0.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No,Yes,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,Site-based - single site,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,20.0,0.0,NaT,NaT,0.0,20.0,20.0,1.0,NaN
3,2025.0,ES,1736 Family Crisis Center,RHY Basic Center Emergency Shelter,Santa Monica,CA,SPA 5 ...,10,2,063384,Yes,Current,Facility-based beds,NaN,0.0,0.0,0.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No,Yes,No,No,No,No,No,No,No,No,No,No,Yes,No,No,No,No,No,No,No,No,No,No,No,No,No,Site-based - single site,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,6.0,0.0,NaT,NaT,0.0,3.0,6.0,0.5,NaN
4,2025.0,RRH,1736 Family Crisis Center,SSVF RRH,Wilmington,CA,SPA 8 ...,15,4,062118,Yes,Current,NaN,NaN,0.0,0.0,9.0,0.0,0.0,0.0,0.0,9.0,0.0,0.0,0.0,No,Yes,No,Yes,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Tenant-based - scattered site,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,9.0,0.0,NaT,NaT,0.0,9.0,9.0,1.0,NaN


In [ ]:
df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")])
print("dropped junk cols; shape now:", df.shape)

dropped junk cols; shape now: (1189, 78)


In [ ]:
df = df.dropna(subset=["Proj. Type", "Total Beds"]).copy()
print("rows after dropping blanks:", len(df))

rows after dropping blanks: 1185


In [ ]:
print(list(df.columns))
df.info()

['Year', 'Proj. Type', 'Organization Name', 'Project Name', 'City', 'State', 'SPA', 'CD', 'SD', 'Geo Code', 'HMIS Participating', 'Inventory Type', 'Bed Type', 'Target Pop.', 'Beds HH w/ Children', 'Units HH w/ Children', 'Beds HH w/o Children', 'Beds HH w/ only Children', 'Veteran Beds HH w/ Children', 'Youth Beds HH w/ Children', 'CH Beds HH w/ Children', 'Veteran Beds HH w/o Children', 'Youth Beds HH w/o Children', 'CH Beds HH w/o Children', 'CH Beds HH w only Children', 'Victim Service Provider', 'Additional Federal Funding?', 'Additional Federal Funding: HUD-VASH', 'Additional Federal Funding: SSVF', 'Additional Federal Funding: GPD-BH', 'Additional Federal Funding: GPD-LD', 'Additional Federal Funding: GPD-HH', 'Additional Federal Funding: GPD-CT', 'Additional Federal Funding: GPD-SITH', 'Additional Federal Funding: GPD-TP', 'Additional Federal Funding: HCHV-CRS', 'Additional Federal Funding: HCHV-SH', 'Additional Federal Funding: BCP', 'Additional Federal Funding: MGH', 'Additio

In [ ]:
import numpy as np

# 1. numeric coercion (bed/unit cols are already float, but harmless)
BED_COLS  = [c for c in df.columns if "bed"  in c.lower()]
UNIT_COLS = [c for c in df.columns if "unit" in c.lower()]
for c in BED_COLS + UNIT_COLS:
    df[c] = pd.to_numeric(df[c].astype(str).str.replace(",", "", regex=False), errors="coerce")

# 2. keep only currently-operating projects
df = df[df["Inventory Type"].astype(str).str.strip().str.lower().eq("current")].copy()

# 3. clean categorical keys
df["spa_num"]   = df["SPA"].astype(str).str.extract(r"(\d+)")[0]
df["Proj. Type"] = df["Proj. Type"].astype(str).str.strip()

# 4. shelter vs permanent housing
df["bed_category"] = np.where(df["Proj. Type"].isin(["ES", "TH", "SH"]), "Shelter", "Permanent Housing")

# 5. the answer: beds by SPA
print("TOTAL beds (current):", int(df["Total Beds"].sum()))
print("\nbeds by SPA:")
print(df.groupby("spa_num")["Total Beds"].sum())
print("\nbeds by SPA x category:")
print(df.groupby(["spa_num", "bed_category"])["Total Beds"].sum().unstack(fill_value=0))

TOTAL beds (current): 71308

beds by SPA:
spa_num
1     2895.0
2     9299.0
3     6634.0
4    26975.0
5     4907.0
6    12137.0
7     3946.0
8     4515.0
Name: Total Beds, dtype: float64

beds by SPA x category:
bed_category  Permanent Housing  Shelter
spa_num                                 
1                        1569.0   1326.0
2                        4926.0   4373.0
3                        5366.0   1268.0
4                       19487.0   7488.0
5                        3696.0   1211.0
6                        6120.0   6017.0
7                        2047.0   1899.0
8                        2241.0   2274.0


In [ ]:
df.to_csv("hic_2025_clean.csv", index=False)
df.groupby("spa_num")["Total Beds"].sum().reset_index(name="total_beds").to_csv("hic_2025_beds_by_spa.csv", index=False)
df.groupby(["spa_num","bed_category"])["Total Beds"].sum().unstack(fill_value=0).reset_index().to_csv("hic_2025_beds_by_spa_category.csv", index=False)
print("saved 3 files")

saved 3 files


In [ ]:
from google.colab import files
files.download("hic_2025_beds_by_spa.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print([c for c in df.columns if "address" in c.lower() or "zip" in c.lower()])
addr = next((c for c in df.columns if "address" in c.lower()), None)
if addr:
    print(df[addr].notna().sum(), "of", len(df), "rows have an address")
    print(df[addr].dropna().head(10).tolist())
else:
    print("no Address column on this sheet")

[]
no Address column on this sheet
